# SupportPilot AI — Model 3: DistilBERT

## Tujuan

Notebook ini digunakan untuk membangun model ketiga
untuk multiclass Intent Classification menggunakan
Transformer-based model DistilBERT.

Berbeda dengan TF-IDF + Logistic Regression dan Linear SVM,
DistilBERT menggunakan contextual representation untuk
merepresentasikan teks berdasarkan konteks token dalam kalimat.

Dataset terdiri dari 46 intent.

Model akan dilatih menggunakan Training Set dan dievaluasi
menggunakan Validation Set.

Test Set tidak digunakan pada tahap model development dan
tetap disimpan untuk final evaluation.

Tahapan:

1. Environment & GPU Check
2. Load Dataset
3. Load Label Mapping
4. Load DistilBERT Tokenizer
5. Token Length Analysis
6. Build PyTorch Dataset
7. Load Pretrained DistilBERT
8. Fine-tuning
9. Validation Evaluation
10. Error Analysis
11. Confidence Analysis
12. Model Comparison

In [1]:
# import library
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

In [2]:
# test environment
SEED = 42

set_seed(SEED)

print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.version.cuda)
print("CUDA aktif   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU          :",
        torch.cuda.get_device_name(0),
    )

PyTorch      : 2.12.0+cu126
CUDA         : 12.6
CUDA aktif   : True
GPU          : NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# path project
TRAIN_PATH = Path(
    "../data/processed/train.csv"
)

VALIDATION_PATH = Path(
    "../data/processed/validation.csv"
)

LABEL_MAPPING_PATH = Path(
    "../data/processed/label_mapping.json"
)

MODEL_OUTPUT_DIR = Path(
    "../models/distilbert_supportpilot"
)

REPORT_DIR = Path(
    "../reports/metrics"
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Train path      :", TRAIN_PATH)
print("Validation path :", VALIDATION_PATH)
print("Label mapping   :", LABEL_MAPPING_PATH)

Train path      : ..\data\processed\train.csv
Validation path : ..\data\processed\validation.csv
Label mapping   : ..\data\processed\label_mapping.json


In [4]:
# validsai file
required_files = [
    TRAIN_PATH,
    VALIDATION_PATH,
    LABEL_MAPPING_PATH,
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {file_path}"
        )

print("✅ Semua file ditemukan.")

✅ Semua file ditemukan.


In [5]:
# load dataset
train_df = pd.read_csv(
    TRAIN_PATH
)

validation_df = pd.read_csv(
    VALIDATION_PATH
)

print(
    f"Train      : {len(train_df):,}"
)

print(
    f"Validation : {len(validation_df):,}"
)

print(
    f"Intent Train      : "
    f"{train_df['intent'].nunique()}"
)

print(
    f"Intent Validation : "
    f"{validation_df['intent'].nunique()}"
)

Train      : 35,861
Validation : 4,483
Intent Train      : 46
Intent Validation : 46


In [6]:
# checck data
train_df[
    [
        "instruction",
        "intent",
        "category",
    ]
].head()

,instruction,intent,category
0,canuhelp me checking the status of my refund,refund_status,RETURNS
1,I'd like to know about ur store opening hours ...,store_opening_hours,STORE
2,would it be possible to modify my fucking onli...,change_order,ORDER
3,"I don't remember my fucking password, can I re...",recover_password,ACCOUNT
4,one of the items ireceived was fucking crushed...,damaged_delivery,DELIVERY


In [7]:
print(
    "Missing instruction Train:",
    train_df["instruction"].isna().sum(),
)

print(
    "Missing intent Train:",
    train_df["intent"].isna().sum(),
)

print(
    "Missing instruction Validation:",
    validation_df["instruction"].isna().sum(),
)

print(
    "Missing intent Validation:",
    validation_df["intent"].isna().sum(),
)

Missing instruction Train: 0
Missing intent Train: 0
Missing instruction Validation: 0
Missing intent Validation: 0


In [8]:
# load label mapping
with open(
    LABEL_MAPPING_PATH,
    "r",
    encoding="utf-8",
) as file:
    mapping_data = json.load(file)

mapping_data

{'label2id': {'add_product': 0,
  'availability': 1,
  'availability_in_store': 2,
  'availability_online': 3,
  'cancel_order': 4,
  'change_account': 5,
  'change_order': 6,
  'close_account': 7,
  'customer_service': 8,
  'damaged_delivery': 9,
  'delivery_issue': 10,
  'delivery_time': 11,
  'exchange_product': 12,
  'exchange_product_in_store': 13,
  'human_agent': 14,
  'missing_item': 15,
  'open_account': 16,
  'order_history': 17,
  'pay': 18,
  'payment_issue': 19,
  'payment_methods': 20,
  'product_information': 21,
  'product_issue': 22,
  'recover_password': 23,
  'refund_policy': 24,
  'refund_status': 25,
  'remove_product': 26,
  'request_invoice': 27,
  'request_refund': 28,
  'request_right_to_rectification': 29,
  'return_policy': 30,
  'return_product': 31,
  'return_product_in_store': 32,
  'return_product_online': 33,
  'sales_period': 34,
  'shipping_costs': 35,
  'store_location': 36,
  'store_opening_hours': 37,
  'submit_feedback': 38,
  'submit_product_feedb

In [9]:
print(
    "Tipe mapping_data:",
    type(mapping_data)
)

print(
    "Top-level keys:",
    mapping_data.keys()
)

Tipe mapping_data: <class 'dict'>
Top-level keys: dict_keys(['label2id', 'id2label'])


In [10]:
# Ambil mapping label -> ID dari file JSON
if "label2id" in mapping_data:
    raw_label2id = mapping_data[
        "label2id"
    ]
else:
    raw_label2id = mapping_data


label2id = {
    str(label): int(idx)
    for label, idx
    in raw_label2id.items()
}


# Buat ulang ID -> label agar format integer konsisten
id2label = {
    idx: label
    for label, idx
    in label2id.items()
}


print(
    "Jumlah label:",
    len(label2id)
)

print(
    "\n10 label pertama:"
)

print(
    list(
        label2id.items()
    )[:10]
)

print(
    "\n10 id2label pertama:"
)

print(
    list(
        id2label.items()
    )[:10]
)

Jumlah label: 46

10 label pertama:
[('add_product', 0), ('availability', 1), ('availability_in_store', 2), ('availability_online', 3), ('cancel_order', 4), ('change_account', 5), ('change_order', 6), ('close_account', 7), ('customer_service', 8), ('damaged_delivery', 9)]

10 id2label pertama:
[(0, 'add_product'), (1, 'availability'), (2, 'availability_in_store'), (3, 'availability_online'), (4, 'cancel_order'), (5, 'change_account'), (6, 'change_order'), (7, 'close_account'), (8, 'customer_service'), (9, 'damaged_delivery')]


In [11]:
# check
train_labels = set(
    train_df["intent"].unique()
)

validation_labels = set(
    validation_df["intent"].unique()
)

mapping_labels = set(
    label2id.keys()
)


missing_train = (
    train_labels
    - mapping_labels
)

missing_validation = (
    validation_labels
    - mapping_labels
)

extra_mapping = (
    mapping_labels
    - train_labels
)


print(
    "Train label tidak ada di mapping:",
    missing_train,
)

print(
    "Validation label tidak ada di mapping:",
    missing_validation,
)

print(
    "Mapping label tidak ada di Train:",
    extra_mapping,
)

Train label tidak ada di mapping: set()
Validation label tidak ada di mapping: set()
Mapping label tidak ada di Train: set()


In [12]:
# konfigurasi base model
MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 64

print("Base model :", MODEL_NAME)
print("Max length :", MAX_LENGTH)

Base model : distilbert/distilbert-base-uncased
Max length : 64


In [13]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "Tokenizer       :",
    tokenizer.__class__.__name__,
)

print(
    "Vocabulary size :",
    tokenizer.vocab_size,
)

print(
    "Model max length:",
    tokenizer.model_max_length,
)

Tokenizer       : BertTokenizer
Vocabulary size : 30522
Model max length: 512


In [14]:
# test satu kalimat
sample_text = "where is my order?"

sample_encoding = tokenizer(
    sample_text,
    add_special_tokens=True,
)

print("Original:")
print(sample_text)

print("\nTokens:")
print(
    tokenizer.convert_ids_to_tokens(
        sample_encoding["input_ids"]
    )
)

print("\nInput IDs:")
print(
    sample_encoding["input_ids"]
)

print("\nAttention Mask:")
print(
    sample_encoding["attention_mask"]
)

Original:
where is my order?

Tokens:
['[CLS]', 'where', 'is', 'my', 'order', '?', '[SEP]']

Input IDs:
[101, 2073, 2003, 2026, 2344, 1029, 102]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1]


In [15]:
train_texts = (
    train_df["instruction"]
    .astype(str)
    .tolist()
)

tokenized_for_length = tokenizer(
    train_texts,
    add_special_tokens=True,
    truncation=False,
    padding=False,
)

train_token_lengths = pd.Series(
    [
        len(input_ids)
        for input_ids
        in tokenized_for_length[
            "input_ids"
        ]
    ]
)

train_token_lengths.describe()

count    35861.000000
mean        15.655001
std          3.543356
min          4.000000
25%         13.000000
50%         16.000000
75%         18.000000
max         29.000000
dtype: float64

In [16]:
max_token_length = int(
    train_token_lengths.max()
)

p95_token_length = float(
    train_token_lengths.quantile(
        0.95
    )
)

p99_token_length = float(
    train_token_lengths.quantile(
        0.99
    )
)

longer_than_max = int(
    (
        train_token_lengths
        > MAX_LENGTH
    ).sum()
)

percentage_longer = (
    longer_than_max
    / len(train_token_lengths)
    * 100
)

print(
    "Maximum token length :",
    max_token_length,
)

print(
    "P95                  :",
    p95_token_length,
)

print(
    "P99                  :",
    p99_token_length,
)

print(
    f"Jumlah > {MAX_LENGTH} token :",
    longer_than_max,
)

print(
    "Persentase           :",
    f"{percentage_longer:.4f}%"
)

Maximum token length : 29
P95                  : 21.0
P99                  : 24.0
Jumlah > 64 token : 0
Persentase           : 0.0000%


In [17]:
# pytorch dataset
class IntentDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        label2id,
        max_length=64,
    ):
        self.texts = (
            texts
            .astype(str)
            .tolist()
        )

        self.labels = (
            labels
            .astype(str)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(
            self.texts
        )

    def __getitem__(
        self,
        idx,
    ):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
        )

        encoding["labels"] = (
            self.label2id[label]
        )

        return encoding

In [18]:
# buat train dan validation dataset
train_dataset = IntentDataset(
    texts=train_df[
        "instruction"
    ],
    labels=train_df[
        "intent"
    ],
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
)

validation_dataset = IntentDataset(
    texts=validation_df[
        "instruction"
    ],
    labels=validation_df[
        "intent"
    ],
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
)

print(
    "Train dataset      :",
    len(train_dataset),
)

print(
    "Validation dataset :",
    len(validation_dataset),
)

Train dataset      : 35861
Validation dataset : 4483


In [19]:
# chceck satu sample secara manual
sample = train_dataset[0]

print("Input IDs:")
print(
    sample["input_ids"]
)

print("\nTokens:")
print(
    tokenizer.convert_ids_to_tokens(
        sample["input_ids"]
    )
)

print(
    "\nLabel ID:",
    sample["labels"],
)

print(
    "Intent hasil mapping:",
    id2label[
        sample["labels"]
    ],
)

print(
    "Intent asli:",
    train_df.iloc[0][
        "intent"
    ],
)

Input IDs:
[101, 2064, 27225, 2884, 2361, 2033, 9361, 1996, 3570, 1997, 2026, 25416, 8630, 102]

Tokens:
['[CLS]', 'can', '##uh', '##el', '##p', 'me', 'checking', 'the', 'status', 'of', 'my', 'ref', '##und', '[SEP]']

Label ID: 25
Intent hasil mapping: refund_status
Intent asli: refund_status


In [20]:
# dynamic padding
data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer
    )
)

print(
    "✅ Data collator siap."
)

✅ Data collator siap.


In [22]:
# load DistilBERT
NUM_LABELS = len(
    label2id
)

print(
    "Jumlah class:",
    NUM_LABELS,
)

Jumlah class: 46


In [23]:
# load pre-trained DistilBERT
model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    )
)

print(
    "Model:",
    model.__class__.__name__,
)

print(
    "Jumlah class:",
    model.config.num_labels,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Install Aplikasi\anaconda\envs\supportpilot-ai\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: DistilBertForSequenceClassification
Jumlah class: 46


In [24]:
# hitung parameter model
total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

print(
    f"Total parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)

Total parameters     : 66,988,846
Trainable parameters : 66,988,846


In [25]:
# GPU sanity check
print(
    "CUDA aktif:",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

    print(
        "VRAM allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            3,
        ),
        "GB",
    )

    print(
        "VRAM reserved:",
        round(
            torch.cuda.memory_reserved()
            / 1024**3,
            3,
        ),
        "GB",
    )

CUDA aktif: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM allocated: 0.0 GB
VRAM reserved: 0.0 GB


In [26]:
# import early stopping 
from transformers import EarlyStoppingCallback

print("✅ EarlyStoppingCallback siap.")

✅ EarlyStoppingCallback siap.


In [27]:
# fungsi evaluasi
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1,
    )

    accuracy = accuracy_score(
        labels,
        predictions,
    )

    macro_precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

In [29]:
import transformers
import inspect

print(
    "Transformers version:",
    transformers.__version__,
)

training_args_params = (
    inspect.signature(
        TrainingArguments.__init__
    ).parameters
)

print(
    "overwrite_output_dir tersedia:",
    "overwrite_output_dir"
    in training_args_params,
)

print(
    "warmup_ratio tersedia:",
    "warmup_ratio"
    in training_args_params,
)

print(
    "warmup_steps tersedia:",
    "warmup_steps"
    in training_args_params,
)

print(
    "eval_strategy tersedia:",
    "eval_strategy"
    in training_args_params,
)

Transformers version: 5.15.0
overwrite_output_dir tersedia: False
warmup_ratio tersedia: False
warmup_steps tersedia: True
eval_strategy tersedia: True


In [30]:
# training configuration
training_args = TrainingArguments(

    # folder untuk checkpoint/model
    output_dir=str(
        MODEL_OUTPUT_DIR
    ),

    # evaluasi setiap selesai 1 epoch
    eval_strategy="epoch",

    # simpan checkpoint setiap selesai 1 epoch
    save_strategy="epoch",

    # logging
    logging_strategy="steps",
    logging_steps=100,

    # learning rate
    learning_rate=2e-5,

    # batch size
    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    # effective batch:
    # 16 × 2 = 32
    gradient_accumulation_steps=2,

    # maksimum epoch
    num_train_epochs=3,

    # regularization
    weight_decay=0.01,

    # scheduler
    lr_scheduler_type="linear",

    # Transformers v5:
    # float 0.10 = warmup 10% dari training steps
    warmup_steps=0.10,

    # mixed precision untuk NVIDIA GPU
    fp16=True,

    # optimizer
    optim="adamw_torch",

    # ambil model terbaik setelah training
    load_best_model_at_end=True,

    # metric utama
    metric_for_best_model="macro_f1",

    greater_is_better=True,

    # maksimal simpan 2 checkpoint
    save_total_limit=2,

    # tidak kirim ke W&B/TensorBoard
    report_to="none",

    # reproducibility
    seed=SEED,
    data_seed=SEED,

    # aman untuk Windows/Jupyter
    dataloader_num_workers=0,
)

print(
    "✅ TrainingArguments berhasil dibuat."
)

✅ TrainingArguments berhasil dibuat.


In [31]:
# pengecekkan konfigurasi
effective_batch_size = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print(
    "Train batch size      :",
    training_args.per_device_train_batch_size,
)

print(
    "Gradient accumulation :",
    training_args.gradient_accumulation_steps,
)

print(
    "Effective batch size  :",
    effective_batch_size,
)

print(
    "Eval batch size       :",
    training_args.per_device_eval_batch_size,
)

print(
    "Learning rate         :",
    training_args.learning_rate,
)

print(
    "Epoch                 :",
    training_args.num_train_epochs,
)

print(
    "Warmup                :",
    training_args.warmup_steps,
)

print(
    "FP16                  :",
    training_args.fp16,
)

print(
    "Best-model metric     :",
    training_args.metric_for_best_model,
)

Train batch size      : 16
Gradient accumulation : 2
Effective batch size  : 32
Eval batch size       : 32
Learning rate         : 2e-05
Epoch                 : 3
Warmup                : 0.1
FP16                  : True
Best-model metric     : macro_f1


In [32]:
# membuat Trainer
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=validation_dataset,

    data_collator=data_collator,

    processing_class=tokenizer,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ],
)

print("✅ Trainer berhasil dibuat.")

✅ Trainer berhasil dibuat.


In [33]:
# check trainer sebelum training
print(
    "Device              :",
    trainer.args.device,
)

print(
    "FP16                :",
    trainer.args.fp16,
)

print(
    "Train records       :",
    len(trainer.train_dataset),
)

print(
    "Validation records  :",
    len(trainer.eval_dataset),
)

print(
    "Metric terbaik      :",
    trainer.args.metric_for_best_model,
)

print(
    "Load best model     :",
    trainer.args.load_best_model_at_end,
)

Device              : cuda:0
FP16                : True
Train records       : 35861
Validation records  : 4483
Metric terbaik      : macro_f1
Load best model     : True


In [34]:
# check satu batch asli
train_dataloader = (
    trainer.get_train_dataloader()
)

sample_batch = next(
    iter(train_dataloader)
)

print(
    "Batch keys:",
    sample_batch.keys(),
)

print(
    "Input IDs shape:",
    sample_batch[
        "input_ids"
    ].shape,
)

print(
    "Attention mask shape:",
    sample_batch[
        "attention_mask"
    ].shape,
)

print(
    "Labels shape:",
    sample_batch[
        "labels"
    ].shape,
)

print(
    "Label minimum:",
    sample_batch[
        "labels"
    ].min().item(),
)

print(
    "Label maximum:",
    sample_batch[
        "labels"
    ].max().item(),
)

Batch keys: KeysView({'input_ids': tensor([[  101,  2026,  4003,  2001,  6430,  2073,  2071,  1045,  2128,  7361,
          5339,  1037,  7909,  3291,   102,     0,     0,     0,     0,     0],
        [  101,  1045,  2288,  2000,  4638,  1996, 11343,  1997,  3688,  3784,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  1045,  2123,  2102,  2066,  2054,  1045, 11829, 11039,  2393,
          2033,  2000,  2156,  2009,   102,     0,     0,     0,     0,     0],
        [  101,  1045,  2031,  2000, 12210,  1996, 11343,  1997,  1037,  8239,
          4031,  1999,  3573,  2129,  2071,  1045,  2079,  2009,   102,     0],
        [  101,  2071,  2017,  2393,  2033,  5587,  6616, 25518,  3688,  2000,
          1996, 11122,  1029,   102,     0,     0,     0,     0,     0,     0],
        [  101,  1045,  1005,  1040,  2066,  2000,  6366,  8239,  5167,  2013,
          1996, 10810,  1010,  2071,  1045,  2131,  2070,  2393,  1029,   102],
        [  

In [35]:
# forward pass satu batch
model.to(
    trainer.args.device
)

batch_on_device = {
    key: value.to(
        trainer.args.device
    )
    for key, value
    in sample_batch.items()
}

model.eval()

with torch.no_grad():

    outputs = model(
        **batch_on_device
    )

print(
    "Loss:",
    outputs.loss.item(),
)

print(
    "Logits shape:",
    outputs.logits.shape,
)

Loss: 3.8238418102264404
Logits shape: torch.Size([16, 46])


In [36]:
# bersihkan vram setelah sanity test
del batch_on_device
del outputs

import gc

gc.collect()

torch.cuda.empty_cache()

print(
    "✅ Sanity test selesai dan cache GPU dibersihkan."
)

✅ Sanity test selesai dan cache GPU dibersihkan.


In [37]:
# mulai fine tunning
training_start = time.perf_counter()

train_result = trainer.train()

training_end = time.perf_counter()

training_time_seconds = (
    training_end
    - training_start
)

print(
    "\n✅ Fine-tuning DistilBERT selesai."
)

print(
    "Training time:",
    round(
        training_time_seconds,
        2,
    ),
    "detik",
)

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.071943,0.033561,0.994200,0.994479,0.994295,0.994290,0.994200
2,0.025287,0.019043,0.995539,0.995749,0.995613,0.995615,0.995540
3,0.029847,0.016997,0.996431,0.996618,0.996491,0.996491,0.996431


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Fine-tuning DistilBERT selesai.
Training time: 292.33 detik


In [38]:
train_result.metrics

{'train_runtime': 291.8775,
 'train_samples_per_second': 368.59,
 'train_steps_per_second': 11.522,
 'total_flos': 609241963194144.0,
 'train_loss': 0.785235912999049,
 'epoch': 3.0}

In [39]:
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)

print(
    "Best Macro F1:",
    trainer.state.best_metric,
)

Best checkpoint: ..\models\distilbert_supportpilot\checkpoint-3363
Best Macro F1: 0.9964905201800945


In [41]:
# check vram
peak_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print(
    "Peak VRAM:",
    round(
        peak_vram_gb,
        2,
    ),
    "GB",
)

Peak VRAM: 1.27 GB


In [44]:
# Step 7H — Simpan model final DistilBERT

from pathlib import Path

# cari project root secara otomatis
current_dir = Path.cwd().resolve()

if (
    (current_dir / "notebooks").exists()
    and
    (current_dir / "models").exists()
):
    PROJECT_ROOT = current_dir

elif (
    current_dir.name == "notebooks"
    and
    (current_dir.parent / "models").exists()
):
    PROJECT_ROOT = current_dir.parent

else:
    raise FileNotFoundError(
        f"Project root tidak ditemukan dari: {current_dir}"
    )

print("Project root :", PROJECT_ROOT)


# lokasi model final
FINAL_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "distilbert_supportpilot"
    / "best_model"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# simpan best model
trainer.save_model(
    str(FINAL_MODEL_DIR)
)


# simpan tokenizer
tokenizer.save_pretrained(
    str(FINAL_MODEL_DIR)
)


print("\n✅ Best DistilBERT berhasil disimpan.")
print("Lokasi:", FINAL_MODEL_DIR)

Project root : C:\Users\LENOVO\Documents\bootcamp\koding data\SupportPilot_AI_Bootcamp_KodingData


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Best DistilBERT berhasil disimpan.
Lokasi: C:\Users\LENOVO\Documents\bootcamp\koding data\SupportPilot_AI_Bootcamp_KodingData\models\distilbert_supportpilot\best_model


In [46]:
# Reload final model untuk verifikasi

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# load tokenizer dari model final
final_tokenizer = AutoTokenizer.from_pretrained(
    str(FINAL_MODEL_DIR)
)

# load model final
final_model = AutoModelForSequenceClassification.from_pretrained(
    str(FINAL_MODEL_DIR)
)

# pindahkan ke GPU
final_model = final_model.to(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("✅ Final model berhasil di-load kembali.")
print("Lokasi model :", FINAL_MODEL_DIR)
print("Model        :", final_model.__class__.__name__)
print("Jumlah class :", final_model.config.num_labels)
print(
    "Device       :",
    next(final_model.parameters()).device,
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Final model berhasil di-load kembali.
Lokasi model : C:\Users\LENOVO\Documents\bootcamp\koding data\SupportPilot_AI_Bootcamp_KodingData\models\distilbert_supportpilot\best_model
Model        : DistilBertForSequenceClassification
Jumlah class : 46
Device       : cuda:0


In [47]:
# Final validation evaluation
# Evaluator khusus model yang sudah di-load dari best_model

final_trainer = Trainer(
    model=final_model,
    args=training_args,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Final Trainer siap.")
print("Validation records :", len(validation_dataset))
print(
    "Device             :",
    next(final_model.parameters()).device,
)

✅ Final Trainer siap.
Validation records : 4483
Device             : cuda:0


In [48]:
# prediksi validation set menggunakan FINAL MODEL

import time

validation_start = time.perf_counter()

validation_output = final_trainer.predict(
    validation_dataset,
    metric_key_prefix="validation",
)

validation_end = time.perf_counter()

validation_wall_time = (
    validation_end - validation_start
)

print("✅ Validation prediction selesai.")
print(
    f"Wall time : "
    f"{validation_wall_time:.2f} detik"
)

✅ Validation prediction selesai.
Wall time : 2.26 detik


In [49]:
# metrics final validation

validation_output.metrics

{'validation_loss': 0.016997456550598145,
 'validation_model_preparation_time': 0.001,
 'validation_accuracy': 0.9964309614097703,
 'validation_macro_precision': 0.9966184860416704,
 'validation_macro_recall': 0.9964909969257796,
 'validation_macro_f1': 0.9964905201800945,
 'validation_weighted_f1': 0.99643094243463,
 'validation_runtime': 2.2537,
 'validation_samples_per_second': 1989.173,
 'validation_steps_per_second': 62.564}

In [50]:
# ringkasan final validation metrics

final_validation_metrics = {
    "accuracy": validation_output.metrics[
        "validation_accuracy"
    ],
    "macro_precision": validation_output.metrics[
        "validation_macro_precision"
    ],
    "macro_recall": validation_output.metrics[
        "validation_macro_recall"
    ],
    "macro_f1": validation_output.metrics[
        "validation_macro_f1"
    ],
    "weighted_f1": validation_output.metrics[
        "validation_weighted_f1"
    ],
    "runtime_seconds": validation_output.metrics[
        "validation_runtime"
    ],
}

final_validation_metrics[
    "average_inference_ms"
] = (
    final_validation_metrics["runtime_seconds"]
    * 1000
    / len(validation_dataset)
)

pd.DataFrame(
    {
        "metric": final_validation_metrics.keys(),
        "value": final_validation_metrics.values(),
    }
)

,metric,value
0,accuracy,0.996431
1,macro_precision,0.996618
2,macro_recall,0.996491
3,macro_f1,0.996491
4,weighted_f1,0.996431
5,runtime_seconds,2.253700
6,average_inference_ms,0.502721


In [51]:
# Error Analysis DistilBERT
# ambil logits, prediction, dan ground truth

import numpy as np
import pandas as pd

val_logits = validation_output.predictions

# antisipasi jika prediction berbentuk tuple
if isinstance(val_logits, tuple):
    val_logits = val_logits[0]

val_true_ids = np.asarray(
    validation_output.label_ids
).astype(int)

val_pred_ids = np.argmax(
    val_logits,
    axis=1,
)


# helper decode label
def decode_label(label_id):
    label_id = int(label_id)

    if label_id in id2label:
        return id2label[label_id]

    return id2label[str(label_id)]


val_true_labels = [
    decode_label(i)
    for i in val_true_ids
]

val_pred_labels = [
    decode_label(i)
    for i in val_pred_ids
]


n_correct = np.sum(
    val_true_ids == val_pred_ids
)

n_error = np.sum(
    val_true_ids != val_pred_ids
)

print("Validation records :", len(val_true_ids))
print("Prediksi benar     :", n_correct)
print("Prediksi salah     :", n_error)
print(
    "Accuracy           :",
    f"{n_correct / len(val_true_ids):.6f}",
)

Validation records : 4483
Prediksi benar     : 4467
Prediksi salah     : 16
Accuracy           : 0.996431


In [52]:
# 10 intent dengan F1 terendah
from sklearn.metrics import classification_report

all_labels = list(label2id.keys())

report_dict = classification_report(
    val_true_labels,
    val_pred_labels,
    labels=all_labels,
    output_dict=True,
    zero_division=0,
)

report_df = (
    pd.DataFrame(report_dict)
    .T
)

intent_report = (
    report_df
    .loc[
        all_labels,
        [
            "precision",
            "recall",
            "f1-score",
            "support",
        ],
    ]
)

lowest_f1 = (
    intent_report
    .sort_values(
        "f1-score",
        ascending=True,
    )
    .head(10)
)

lowest_f1

,precision,recall,f1-score,support
track_order,1.000000,0.929293,0.963351,99.0
track_delivery,0.934579,1.000000,0.966184,100.0
return_policy,0.979592,0.969697,0.974619,99.0
return_product_online,0.970588,1.000000,0.985075,99.0
availability,0.979592,1.000000,0.989691,96.0
return_product_in_store,1.000000,0.989899,0.994924,99.0
product_information,1.000000,0.989899,0.994924,99.0
return_product,1.000000,0.989899,0.994924,99.0
submit_feedback,1.000000,0.989899,0.994924,99.0
refund_policy,1.000000,0.990000,0.994975,100.0


In [53]:
# cari confusion pair terbesar
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    val_true_labels,
    val_pred_labels,
    labels=all_labels,
)

confusion_pairs = []

for i, actual_label in enumerate(all_labels):
    for j, predicted_label in enumerate(all_labels):

        if (
            i != j
            and cm[i, j] > 0
        ):
            confusion_pairs.append(
                {
                    "actual_intent": actual_label,
                    "predicted_intent": predicted_label,
                    "count": int(cm[i, j]),
                }
            )


confusion_df = (
    pd.DataFrame(confusion_pairs)
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(drop=True)
)

confusion_df.head(15)

,actual_intent,predicted_intent,count
0,track_order,track_delivery,7
1,return_policy,return_product_online,3
2,product_information,availability,1
3,human_agent,availability,1
4,refund_policy,return_policy,1
5,return_product,cancel_order,1
6,return_product_in_store,return_policy,1
7,submit_feedback,submit_product_feedback,1


In [56]:
# tampilkan semua prediction error

import numpy as np
import pandas as pd


# =========================================================
# 1. Cari teks asli validation dari variabel sebelum tokenisasi
# =========================================================

source_validation_texts = None

# opsi 1 — validation_df
if (
    "validation_df" in globals()
    and isinstance(validation_df, pd.DataFrame)
    and "instruction" in validation_df.columns
):
    source_validation_texts = (
        validation_df["instruction"]
        .astype(str)
        .tolist()
    )

# opsi 2 — val_df
elif (
    "val_df" in globals()
    and isinstance(val_df, pd.DataFrame)
    and "instruction" in val_df.columns
):
    source_validation_texts = (
        val_df["instruction"]
        .astype(str)
        .tolist()
    )

# opsi 3 — X_validation
elif "X_validation" in globals():

    if isinstance(X_validation, pd.Series):
        source_validation_texts = (
            X_validation
            .astype(str)
            .tolist()
        )

    else:
        source_validation_texts = [
            str(x)
            for x in X_validation
        ]

# opsi 4 — validation_texts jika sebelumnya memang sudah dibuat
elif "validation_texts" in globals():

    source_validation_texts = [
        str(x)
        for x in validation_texts
    ]


# =========================================================
# 2. Pastikan jumlah teks sama dengan prediction
# =========================================================

if source_validation_texts is None:

    print(
        "⚠️ Teks asli validation belum ditemukan."
    )

    source_validation_texts = [
        "(teks tidak tersedia)"
    ] * len(val_true_ids)

elif len(source_validation_texts) != len(val_true_ids):

    print(
        "⚠️ Jumlah teks validation tidak sama "
        "dengan jumlah prediction."
    )

    print(
        "Jumlah teks      :",
        len(source_validation_texts),
    )

    print(
        "Jumlah prediction:",
        len(val_true_ids),
    )

    source_validation_texts = [
        "(teks tidak tersedia)"
    ] * len(val_true_ids)


# =========================================================
# 3. Cari index yang salah prediksi
# =========================================================

error_indices = np.where(
    val_true_ids != val_pred_ids
)[0]


# =========================================================
# 4. Buat dataframe error
# =========================================================

error_df = pd.DataFrame(
    {
        "row": error_indices,

        "instruction": [
            source_validation_texts[i]
            for i in error_indices
        ],

        "actual_intent": [
            val_true_labels[i]
            for i in error_indices
        ],

        "predicted_intent": [
            val_pred_labels[i]
            for i in error_indices
        ],
    }
)


print(
    "Total prediction error:",
    len(error_df),
)

error_df

Total prediction error: 16


,row,instruction,actual_intent,predicted_intent
0,232,"I ogt to see where my shipment is, how can I d...",track_order,track_delivery
1,291,what are the produt characteristics,product_information,availability
2,520,how can I see where my package is?,track_order,track_delivery
3,552,how could i rturn a fucking item online,return_policy,return_product_online
4,583,i want assistance to check where my shpiment is,track_order,track_delivery
5,833,how to return a fucking item online,return_policy,return_product_online
6,850,need to rturn an order what should i do,return_product,cancel_order
7,997,how to return an item in stroe,return_product_in_store,return_policy
8,1045,"I would like to find my oder, I need help",track_order,track_delivery
9,2017,can i return a fucking prduct online,return_policy,return_product_online


In [57]:
# Confidence analysis

import numpy as np
import pandas as pd
import torch


# =========================================================
# 1. Ambil logits hasil validation
# =========================================================

validation_logits = validation_output.predictions

print(
    "Logits shape:",
    validation_logits.shape,
)


# =========================================================
# 2. Ubah logits menjadi probability
# =========================================================

validation_probabilities = torch.softmax(
    torch.tensor(validation_logits),
    dim=1,
).numpy()


# =========================================================
# 3. Confidence prediction tertinggi
# =========================================================

predicted_confidence = (
    validation_probabilities.max(axis=1)
)

predicted_ids_from_prob = (
    validation_probabilities.argmax(axis=1)
)


# =========================================================
# 4. Ambil Top-2 prediction untuk menghitung margin
# =========================================================

sorted_probability_indices = np.argsort(
    validation_probabilities,
    axis=1,
)

second_best_ids = (
    sorted_probability_indices[:, -2]
)

second_best_confidence = (
    validation_probabilities[
        np.arange(
            len(validation_probabilities)
        ),
        second_best_ids,
    ]
)

confidence_margin = (
    predicted_confidence
    - second_best_confidence
)


# =========================================================
# 5. Confidence pada actual class
# =========================================================

actual_confidence = (
    validation_probabilities[
        np.arange(
            len(validation_probabilities)
        ),
        val_true_ids,
    ]
)


# =========================================================
# 6. Buat dataframe semua validation
# =========================================================

confidence_df = pd.DataFrame(
    {
        "row": np.arange(
            len(val_true_ids)
        ),

        "actual_intent": [
            id2label[int(i)]
            for i in val_true_ids
        ],

        "predicted_intent": [
            id2label[int(i)]
            for i in predicted_ids_from_prob
        ],

        "predicted_confidence":
            predicted_confidence,

        "actual_confidence":
            actual_confidence,

        "second_best_intent": [
            id2label[int(i)]
            for i in second_best_ids
        ],

        "second_best_confidence":
            second_best_confidence,

        "confidence_margin":
            confidence_margin,

        "correct":
            val_true_ids
            == predicted_ids_from_prob,
    }
)


print(
    "Jumlah validation:",
    len(confidence_df),
)

print(
    "Jumlah benar:",
    confidence_df["correct"].sum(),
)

print(
    "Jumlah salah:",
    (~confidence_df["correct"]).sum(),
)

confidence_df.head()

Logits shape: (4483, 46)
Jumlah validation: 4483
Jumlah benar: 4467
Jumlah salah: 16


,row,actual_intent,predicted_intent,predicted_confidence,actual_confidence,second_best_intent,second_best_confidence,confidence_margin,correct
0,0,payment_methods,payment_methods,0.998082,0.998082,refund_policy,0.000415,0.997667,True
1,1,product_information,product_information,0.999167,0.999167,return_product_in_store,0.000063,0.999105,True
2,2,recover_password,recover_password,0.999033,0.999033,close_account,0.000073,0.998959,True
3,3,request_right_to_rectification,request_right_to_rectification,0.999038,0.999038,track_order,0.000087,0.998951,True
4,4,missing_item,missing_item,0.998311,0.998311,damaged_delivery,0.000187,0.998124,True


In [58]:
# statistik confidence model

confidence_summary = (
    confidence_df[
        [
            "predicted_confidence",
            "actual_confidence",
            "second_best_confidence",
            "confidence_margin",
        ]
    ]
    .describe()
)

confidence_summary

,predicted_confidence,actual_confidence,second_best_confidence,confidence_margin
count,4483.000000,4483.000000,4483.000000,4483.000000
mean,0.997318,0.994448,0.001499,0.995820
std,0.019195,0.056911,0.017239,0.036329
min,0.469671,0.000191,0.000046,0.013560
25%,0.998878,0.998878,0.000091,0.998729
50%,0.998991,0.998991,0.000115,0.998880
75%,0.999059,0.999059,0.000149,0.998958
max,0.999316,0.999316,0.473170,0.999260


In [59]:
# confidence khusus prediction yang salah

error_confidence_df = (
    confidence_df[
        confidence_df["correct"] == False
    ]
    .copy()
)

error_confidence_df[
    "instruction"
] = [
    source_validation_texts[i]
    for i in error_confidence_df["row"]
]


error_confidence_df = (
    error_confidence_df[
        [
            "row",
            "instruction",
            "actual_intent",
            "predicted_intent",
            "predicted_confidence",
            "actual_confidence",
            "second_best_intent",
            "second_best_confidence",
            "confidence_margin",
        ]
    ]
    .sort_values(
        "predicted_confidence",
        ascending=False,
    )
    .reset_index(drop=True)
)


error_confidence_df

,row,instruction,actual_intent,predicted_intent,predicted_confidence,actual_confidence,second_best_intent,second_best_confidence,confidence_margin
0,552,how could i rturn a fucking item online,return_policy,return_product_online,0.998303,0.000191,availability_online,0.000238,0.998066
1,2017,can i return a fucking prduct online,return_policy,return_product_online,0.996800,0.001692,return_policy,0.001692,0.995109
2,997,how to return an item in stroe,return_product_in_store,return_policy,0.995439,0.001385,return_product_in_store,0.001385,0.994054
3,1045,"I would like to find my oder, I need help",track_order,track_delivery,0.995243,0.003268,track_order,0.003268,0.991975
4,3631,"I want to see were my shipment is, how can I d...",track_order,track_delivery,0.981363,0.016951,track_order,0.016951,0.964412
5,520,how can I see where my package is?,track_order,track_delivery,0.972625,0.025319,track_order,0.025319,0.947306
6,850,need to rturn an order what should i do,return_product,cancel_order,0.970680,0.021740,return_product,0.021740,0.948940
7,4310,is it possible to see where my shipment is ?,track_order,track_delivery,0.967329,0.030613,track_order,0.030613,0.936716
8,833,how to return a fucking item online,return_policy,return_product_online,0.965501,0.030734,return_policy,0.030734,0.934766
9,232,"I ogt to see where my shipment is, how can I d...",track_order,track_delivery,0.965292,0.032646,track_order,0.032646,0.932646


In [60]:
# Per-class performance

from sklearn.metrics import classification_report
import pandas as pd


# urutan label
label_ids = list(range(NUM_LABELS))

label_names = [
    id2label[i]
    for i in label_ids
]


# classification report validation
validation_classification_report = classification_report(
    val_true_ids,
    predicted_ids_from_prob,
    labels=label_ids,
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)


# ubah ke dataframe
per_class_df = (
    pd.DataFrame(
        validation_classification_report
    )
    .T
)


# hanya class intent
per_class_df = (
    per_class_df
    .loc[label_names]
    .copy()
)


# urutkan berdasarkan F1 terendah
lowest_f1_df = (
    per_class_df
    .sort_values(
        "f1-score",
        ascending=True,
    )
    .head(10)
)


print(
    "10 Intent dengan F1 terendah"
)

lowest_f1_df

10 Intent dengan F1 terendah


,precision,recall,f1-score,support
track_order,1.000000,0.929293,0.963351,99.0
track_delivery,0.934579,1.000000,0.966184,100.0
return_policy,0.979592,0.969697,0.974619,99.0
return_product_online,0.970588,1.000000,0.985075,99.0
availability,0.979592,1.000000,0.989691,96.0
return_product_in_store,1.000000,0.989899,0.994924,99.0
product_information,1.000000,0.989899,0.994924,99.0
return_product,1.000000,0.989899,0.994924,99.0
submit_feedback,1.000000,0.989899,0.994924,99.0
refund_policy,1.000000,0.990000,0.994975,100.0


In [61]:
# confusion pair validation DistilBERT

error_pairs_df = (
    confidence_df[
        confidence_df["correct"] == False
    ]
    .groupby(
        [
            "actual_intent",
            "predicted_intent",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "Jumlah jenis confusion pair:",
    len(error_pairs_df),
)


print(
    "\nTop 15 Confusion Pair:"
)


error_pairs_df.head(15)

Jumlah jenis confusion pair: 8

Top 15 Confusion Pair:


,actual_intent,predicted_intent,count
0,track_order,track_delivery,7
1,return_policy,return_product_online,3
2,product_information,availability,1
3,human_agent,availability,1
4,refund_policy,return_policy,1
5,return_product,cancel_order,1
6,return_product_in_store,return_policy,1
7,submit_feedback,submit_product_feedback,1


In [62]:
# analisis konsentrasi error

total_errors = len(
    confidence_df[
        confidence_df["correct"] == False
    ]
)


top_confusion_count = (
    error_pairs_df
    .head(2)["count"]
    .sum()
)


top_confusion_percentage = (
    top_confusion_count
    / total_errors
    * 100
)


high_confidence_errors = (
    error_confidence_df[
        error_confidence_df[
            "predicted_confidence"
        ] >= 0.90
    ]
)


high_confidence_error_percentage = (
    len(high_confidence_errors)
    / total_errors
    * 100
)


print(
    f"Total prediction error     : {total_errors}"
)

print(
    f"Top-2 confusion error      : {top_confusion_count}"
)

print(
    "Persentase Top-2 confusion : "
    f"{top_confusion_percentage:.2f}%"
)

print()

print(
    "Error confidence >= 90%    :",
    len(high_confidence_errors),
)

print(
    "Persentase overconfident   : "
    f"{high_confidence_error_percentage:.2f}%"
)

Total prediction error     : 16
Top-2 confusion error      : 10
Persentase Top-2 confusion : 62.50%

Error confidence >= 90%    : 13
Persentase overconfident   : 81.25%


In [63]:
# Final Validation Model Comparison

import pandas as pd


final_validation_comparison = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "accuracy": 0.982378,
            "macro_f1": 0.982562,
            "inference_ms": 0.014273,
        },
        {
            "model": "Linear SVM",
            "accuracy": 0.987285,
            "macro_f1": 0.987429,
            "inference_ms": 0.052467,
        },
        {
            "model": "DistilBERT",
            "accuracy": 0.996431,
            "macro_f1": 0.996491,
            "inference_ms": 0.502721,
        },
    ]
)


final_validation_comparison

,model,accuracy,macro_f1,inference_ms
0,Logistic Regression,0.982378,0.982562,0.014273
1,Linear SVM,0.987285,0.987429,0.052467
2,DistilBERT,0.996431,0.996491,0.502721


In [64]:
# improvement model

logreg_accuracy = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "Logistic Regression",
        "accuracy",
    ]
    .iloc[0]
)

svm_accuracy = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "Linear SVM",
        "accuracy",
    ]
    .iloc[0]
)

distilbert_accuracy = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "DistilBERT",
        "accuracy",
    ]
    .iloc[0]
)


logreg_f1 = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "Logistic Regression",
        "macro_f1",
    ]
    .iloc[0]
)

svm_f1 = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "Linear SVM",
        "macro_f1",
    ]
    .iloc[0]
)

distilbert_f1 = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "DistilBERT",
        "macro_f1",
    ]
    .iloc[0]
)


print("IMPROVEMENT VALIDATION")
print("=" * 50)

print(
    "Linear SVM vs Logistic Regression"
)

print(
    f"Accuracy : "
    f"{(svm_accuracy - logreg_accuracy) * 100:.4f} percentage point"
)

print(
    f"Macro F1 : "
    f"{(svm_f1 - logreg_f1) * 100:.4f} percentage point"
)


print("\nDistilBERT vs Linear SVM")

print(
    f"Accuracy : "
    f"{(distilbert_accuracy - svm_accuracy) * 100:.4f} percentage point"
)

print(
    f"Macro F1 : "
    f"{(distilbert_f1 - svm_f1) * 100:.4f} percentage point"
)


print("\nDistilBERT vs Logistic Regression")

print(
    f"Accuracy : "
    f"{(distilbert_accuracy - logreg_accuracy) * 100:.4f} percentage point"
)

print(
    f"Macro F1 : "
    f"{(distilbert_f1 - logreg_f1) * 100:.4f} percentage point"
)

IMPROVEMENT VALIDATION
Linear SVM vs Logistic Regression
Accuracy : 0.4907 percentage point
Macro F1 : 0.4867 percentage point

DistilBERT vs Linear SVM
Accuracy : 0.9146 percentage point
Macro F1 : 0.9062 percentage point

DistilBERT vs Logistic Regression
Accuracy : 1.4053 percentage point
Macro F1 : 1.3929 percentage point


In [65]:
# jumlah prediction error tiap model

validation_size = 4483


final_validation_comparison[
    "correct_predictions"
] = (
    final_validation_comparison["accuracy"]
    * validation_size
).round().astype(int)


final_validation_comparison[
    "prediction_errors"
] = (
    validation_size
    - final_validation_comparison[
        "correct_predictions"
    ]
)


final_validation_comparison[
    [
        "model",
        "correct_predictions",
        "prediction_errors",
    ]
]

,model,correct_predictions,prediction_errors
0,Logistic Regression,4404,79
1,Linear SVM,4426,57
2,DistilBERT,4467,16


In [66]:
# error reduction

logreg_errors = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "Logistic Regression",
        "prediction_errors",
    ]
    .iloc[0]
)

svm_errors = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "Linear SVM",
        "prediction_errors",
    ]
    .iloc[0]
)

distilbert_errors = (
    final_validation_comparison
    .loc[
        final_validation_comparison["model"]
        == "DistilBERT",
        "prediction_errors",
    ]
    .iloc[0]
)


error_reduction_vs_logreg = (
    (logreg_errors - distilbert_errors)
    / logreg_errors
    * 100
)

error_reduction_vs_svm = (
    (svm_errors - distilbert_errors)
    / svm_errors
    * 100
)


print(
    "Error Logistic Regression :",
    logreg_errors,
)

print(
    "Error Linear SVM          :",
    svm_errors,
)

print(
    "Error DistilBERT          :",
    distilbert_errors,
)

print()

print(
    "DistilBERT error reduction vs Logistic Regression:"
)

print(
    f"{error_reduction_vs_logreg:.2f}%"
)

print()

print(
    "DistilBERT error reduction vs Linear SVM:"
)

print(
    f"{error_reduction_vs_svm:.2f}%"
)

Error Logistic Regression : 79
Error Linear SVM          : 57
Error DistilBERT          : 16

DistilBERT error reduction vs Logistic Regression:
79.75%

DistilBERT error reduction vs Linear SVM:
71.93%


In [67]:
# pilih model berdasarkan Macro F1 validation

winner = (
    final_validation_comparison
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0]
)


print("FINAL MODEL SELECTION")
print("=" * 50)

print(
    "Model terpilih :",
    winner["model"],
)

print(
    f"Accuracy       : "
    f"{winner['accuracy']:.6f}"
)

print(
    f"Macro F1       : "
    f"{winner['macro_f1']:.6f}"
)

print()

print(
    "Selection metric : Macro F1"
)

print(
    "Selection data   : Validation Set"
)

print(
    "Test Set         : BELUM DIGUNAKAN"
)

FINAL MODEL SELECTION
Model terpilih : DistilBERT
Accuracy       : 0.996431
Macro F1       : 0.996491

Selection metric : Macro F1
Selection data   : Validation Set
Test Set         : BELUM DIGUNAKAN


In [68]:
#  Lock final model sebelum Test Set

FINAL_SELECTED_MODEL = "DistilBERT"
FINAL_SELECTION_METRIC = "Macro F1"
FINAL_VALIDATION_MACRO_F1 = 0.996491

final_model.eval()

print("FINAL MODEL LOCKED")
print("=" * 50)
print("Model              :", FINAL_SELECTED_MODEL)
print("Selection metric   :", FINAL_SELECTION_METRIC)
print(
    "Validation Macro F1:",
    FINAL_VALIDATION_MACRO_F1,
)
print("Model mode         : evaluation")
print()
print("Mulai tahap ini model tidak dituning lagi.")

FINAL MODEL LOCKED
Model              : DistilBERT
Selection metric   : Macro F1
Validation Macro F1: 0.996491
Model mode         : evaluation

Mulai tahap ini model tidak dituning lagi.


In [69]:
# Cari dataframe Test Set

candidate_test_vars = [
    "test_df",
    "df_test",
    "test_data",
    "test",
]

found_test_vars = []

for var_name in candidate_test_vars:

    if var_name in globals():

        obj = globals()[var_name]

        if isinstance(obj, pd.DataFrame):

            found_test_vars.append(
                (
                    var_name,
                    obj.shape,
                    list(obj.columns),
                )
            )


print("Candidate Test DataFrame:")
print("=" * 50)

if found_test_vars:

    for item in found_test_vars:
        print(
            f"{item[0]:15s}",
            "shape =",
            item[1],
            "| columns =",
            item[2],
        )

else:
    print(
        "Test DataFrame belum ditemukan "
        "di memory notebook."
    )

Candidate Test DataFrame:
Test DataFrame belum ditemukan di memory notebook.


In [70]:
# Load FINAL Test Set
# Test Set hanya dimuat, BELUM dilakukan inference

from pathlib import Path
import pandas as pd


# cari project root secara aman
CURRENT_DIR = Path.cwd().resolve()

candidate_roots = [
    CURRENT_DIR,
    CURRENT_DIR.parent,
]

PROJECT_ROOT = None

for root in candidate_roots:

    candidate_test_path = (
        root
        / "data"
        / "processed"
        / "test.csv"
    )

    if candidate_test_path.exists():

        PROJECT_ROOT = root
        break


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "data/processed/test.csv tidak ditemukan."
    )


TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "test.csv"
)


test_df = pd.read_csv(TEST_PATH)


print("FINAL TEST SET LOADED")
print("=" * 55)

print(
    "Project root :",
    PROJECT_ROOT,
)

print(
    "Test path    :",
    TEST_PATH,
)

print(
    "Shape        :",
    test_df.shape,
)

print(
    "Columns      :",
    list(test_df.columns),
)

print()
print("🔒 Belum dilakukan prediction/inference.")

FINAL TEST SET LOADED
Project root : C:\Users\LENOVO\Documents\bootcamp\koding data\SupportPilot_AI_Bootcamp_KodingData
Test path    : C:\Users\LENOVO\Documents\bootcamp\koding data\SupportPilot_AI_Bootcamp_KodingData\data\processed\test.csv
Shape        : (4483, 5)
Columns      : ['instruction', 'intent', 'category', 'tags', 'response']

🔒 Belum dilakukan prediction/inference.


In [71]:
# Validasi FINAL Test Set
# Tidak melihat distribusi label / performa

required_columns = {
    "instruction",
    "intent",
}


missing_columns = (
    required_columns
    - set(test_df.columns)
)


assert not missing_columns, (
    f"Kolom wajib tidak ditemukan: "
    f"{missing_columns}"
)


duplicate_rows = (
    test_df
    .duplicated()
    .sum()
)


missing_text = (
    test_df["instruction"]
    .isna()
    .sum()
)


missing_label = (
    test_df["intent"]
    .isna()
    .sum()
)


print("FINAL TEST SET CHECK")
print("=" * 55)

print(
    "Jumlah record :",
    len(test_df),
)

print(
    "Jumlah kolom  :",
    len(test_df.columns),
)

print(
    "Missing text  :",
    missing_text,
)

print(
    "Missing label :",
    missing_label,
)

print(
    "Duplicate row :",
    duplicate_rows,
)

print()
print("✅ Struktur Test Set siap.")
print("🔒 Test inference BELUM dijalankan.")

FINAL TEST SET CHECK
Jumlah record : 4483
Jumlah kolom  : 5
Missing text  : 0
Missing label : 0
Duplicate row : 0

✅ Struktur Test Set siap.
🔒 Test inference BELUM dijalankan.


In [72]:
# Data Leakage Check
# Pastikan Test Set tidak overlap dengan Train / Validation

train_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train.csv"
)

validation_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validation.csv"
)


train_check_df = pd.read_csv(train_path)
validation_check_df = pd.read_csv(validation_path)


# normalisasi ringan untuk pengecekan teks
def normalize_check_text(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


train_texts_check = set(
    normalize_check_text(
        train_check_df["instruction"]
    )
)

validation_texts_check = set(
    normalize_check_text(
        validation_check_df["instruction"]
    )
)

test_texts_check = set(
    normalize_check_text(
        test_df["instruction"]
    )
)


train_test_overlap = (
    train_texts_check
    & test_texts_check
)

validation_test_overlap = (
    validation_texts_check
    & test_texts_check
)


print("DATA LEAKAGE CHECK")
print("=" * 55)

print(
    "Train records      :",
    len(train_check_df),
)

print(
    "Validation records :",
    len(validation_check_df),
)

print(
    "Test records       :",
    len(test_df),
)

print()

print(
    "Train ↔ Test overlap      :",
    len(train_test_overlap),
)

print(
    "Validation ↔ Test overlap :",
    len(validation_test_overlap),
)


if (
    len(train_test_overlap) == 0
    and
    len(validation_test_overlap) == 0
):
    print()
    print("✅ Tidak ditemukan data leakage.")
    print("✅ Final Test Set aman digunakan.")

else:
    print()
    print("❌ WARNING: ditemukan overlap dataset.")
    print("JANGAN lakukan final inference dulu.")

DATA LEAKAGE CHECK
Train records      : 35861
Validation records : 4483
Test records       : 4483

Train ↔ Test overlap      : 0
Validation ↔ Test overlap : 0

✅ Tidak ditemukan data leakage.
✅ Final Test Set aman digunakan.


In [73]:
# Preflight FINAL Test Set
# Belum melakukan inference

test_labels_text = (
    test_df["intent"]
    .astype(str)
    .tolist()
)

unknown_test_labels = sorted(
    set(test_labels_text)
    - set(label2id.keys())
)

print("FINAL TEST LABEL CHECK")
print("=" * 55)

print(
    "Jumlah record :",
    len(test_df),
)

print(
    "Jumlah class  :",
    test_df["intent"].nunique(),
)

print(
    "Unknown label :",
    unknown_test_labels,
)

if len(unknown_test_labels) == 0:
    print()
    print("✅ Semua label Test Set valid.")
    print("✅ Siap FINAL inference.")
else:
    print()
    print("❌ Ada label yang tidak terdapat di label2id.")
    print("JANGAN lakukan inference.")

FINAL TEST LABEL CHECK
Jumlah record : 4483
Jumlah class  : 46
Unknown label : []

✅ Semua label Test Set valid.
✅ Siap FINAL inference.


In [74]:
# ============================================================
#  FINAL TEST INFERENCE
# ============================================================

import time
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


# ------------------------------------------------------------
# 1. Persiapkan FINAL Test Set
# ------------------------------------------------------------

test_texts = (
    test_df["instruction"]
    .astype(str)
    .tolist()
)

y_test = np.array(
    [
        label2id[label]
        for label in test_df["intent"].astype(str)
    ]
)


# ------------------------------------------------------------
# 2. Konfigurasi inference
# ------------------------------------------------------------

TEST_BATCH_SIZE = 32

TEST_MAX_LENGTH = (
    MAX_LENGTH
    if "MAX_LENGTH" in globals()
    else 64
)

device = next(
    final_model.parameters()
).device

final_model.eval()


print("FINAL TEST INFERENCE")
print("=" * 60)

print(
    "Records        :",
    len(test_texts),
)

print(
    "Batch size     :",
    TEST_BATCH_SIZE,
)

print(
    "Max length     :",
    TEST_MAX_LENGTH,
)

print(
    "Device         :",
    device,
)

print()
print(
    "🔒 FINAL Test Set inference dimulai..."
)


# ------------------------------------------------------------
# 3. Prediction
# ------------------------------------------------------------

all_logits = []

if torch.cuda.is_available():
    torch.cuda.synchronize()


test_start = time.perf_counter()


with torch.inference_mode():

    for start_idx in range(
        0,
        len(test_texts),
        TEST_BATCH_SIZE,
    ):

        batch_texts = test_texts[
            start_idx:
            start_idx + TEST_BATCH_SIZE
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=TEST_MAX_LENGTH,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }

        outputs = final_model(
            **encoded
        )

        all_logits.append(
            outputs.logits
            .detach()
            .cpu()
        )


if torch.cuda.is_available():
    torch.cuda.synchronize()


test_end = time.perf_counter()


# ------------------------------------------------------------
# 4. Gabungkan prediction
# ------------------------------------------------------------

test_logits = torch.cat(
    all_logits,
    dim=0,
)

y_test_pred = (
    torch.argmax(
        test_logits,
        dim=1,
    )
    .numpy()
)


# ------------------------------------------------------------
# 5. Hitung inference time
# ------------------------------------------------------------

test_inference_seconds = (
    test_end
    - test_start
)

test_average_inference_ms = (
    test_inference_seconds
    / len(test_texts)
    * 1000
)


# ------------------------------------------------------------
# 6. Metrics FINAL
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    y_test,
    y_test_pred,
)

test_macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0,
)

test_macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0,
)

test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0,
)

test_weighted_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted",
    zero_division=0,
)


# ------------------------------------------------------------
# 7. Jumlah correct / error
# ------------------------------------------------------------

test_correct = int(
    (y_test == y_test_pred).sum()
)

test_errors = int(
    (y_test != y_test_pred).sum()
)


print()
print(
    "✅ FINAL TEST INFERENCE SELESAI"
)

print("=" * 60)

print(
    f"Accuracy        : {test_accuracy:.6f}"
)

print(
    f"Macro Precision : {test_macro_precision:.6f}"
)

print(
    f"Macro Recall    : {test_macro_recall:.6f}"
)

print(
    f"Macro F1        : {test_macro_f1:.6f}"
)

print(
    f"Weighted F1     : {test_weighted_f1:.6f}"
)

print()

print(
    f"Correct         : {test_correct}"
)

print(
    f"Error           : {test_errors}"
)

print()

print(
    f"Inference time  : "
    f"{test_inference_seconds:.4f} detik"
)

print(
    f"Average/sample  : "
    f"{test_average_inference_ms:.4f} ms"
)

FINAL TEST INFERENCE
Records        : 4483
Batch size     : 32
Max length     : 64
Device         : cuda:0

🔒 FINAL Test Set inference dimulai...

✅ FINAL TEST INFERENCE SELESAI
Accuracy        : 0.997323
Macro Precision : 0.997436
Macro Recall    : 0.997287
Macro F1        : 0.997325
Weighted F1     : 0.997326

Correct         : 4471
Error           : 12

Inference time  : 5.4182 detik
Average/sample  : 1.2086 ms


In [75]:
# ============================================================
#  FINAL TEST CLASSIFICATION REPORT
# ============================================================

import pandas as pd
from sklearn.metrics import classification_report


# ------------------------------------------------------------
# Buat urutan nama class yang aman
# ------------------------------------------------------------

class_ids = list(range(NUM_LABELS))

class_names = [
    id2label[i]
    if i in id2label
    else id2label[str(i)]
    for i in class_ids
]


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

test_report_dict = classification_report(
    y_test,
    y_test_pred,
    labels=class_ids,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)


# ------------------------------------------------------------
# Ambil hanya baris per intent
# ------------------------------------------------------------

test_classification_df = (
    pd.DataFrame(test_report_dict)
    .T
    .loc[class_names]
)


print("FINAL TEST CLASSIFICATION REPORT")
print("=" * 70)

display(
    test_classification_df
    .sort_values(
        "f1-score",
        ascending=True,
    )
)


# ------------------------------------------------------------
# 10 Intent F1 terendah
# ------------------------------------------------------------

lowest_10_test_f1 = (
    test_classification_df
    .sort_values(
        "f1-score",
        ascending=True,
    )
    .head(10)
)


print()
print("10 INTENT DENGAN F1 TERENDAH — FINAL TEST")
print("=" * 70)

display(
    lowest_10_test_f1[
        [
            "precision",
            "recall",
            "f1-score",
            "support",
        ]
    ]
)

FINAL TEST CLASSIFICATION REPORT


,precision,recall,f1-score,support
wrong_item,0.961538,1.000000,0.980392,100.0
track_order,1.000000,0.969388,0.984456,98.0
damaged_delivery,1.000000,0.969697,0.984615,99.0
return_policy,1.000000,0.970000,0.984772,100.0
track_delivery,0.970588,1.000000,0.985075,99.0
return_product_online,0.970588,1.000000,0.985075,99.0
missing_item,1.000000,0.986111,0.993007,72.0
store_location,0.989247,1.000000,0.994595,92.0
return_product,1.000000,0.990000,0.994975,100.0
request_invoice,1.000000,0.990000,0.994975,100.0



10 INTENT DENGAN F1 TERENDAH — FINAL TEST


,precision,recall,f1-score,support
wrong_item,0.961538,1.000000,0.980392,100.0
track_order,1.000000,0.969388,0.984456,98.0
damaged_delivery,1.000000,0.969697,0.984615,99.0
return_policy,1.000000,0.970000,0.984772,100.0
track_delivery,0.970588,1.000000,0.985075,99.0
return_product_online,0.970588,1.000000,0.985075,99.0
missing_item,1.000000,0.986111,0.993007,72.0
store_location,0.989247,1.000000,0.994595,92.0
return_product,1.000000,0.990000,0.994975,100.0
request_invoice,1.000000,0.990000,0.994975,100.0


In [76]:
# ============================================================
#  FINAL TEST CONFUSION PAIRS
# ============================================================

test_error_mask = (
    y_test
    != y_test_pred
)


test_confusion_df = pd.DataFrame(
    {
        "actual_intent": [
            id2label[int(label)]
            if int(label) in id2label
            else id2label[str(int(label))]
            for label in y_test[test_error_mask]
        ],

        "predicted_intent": [
            id2label[int(label)]
            if int(label) in id2label
            else id2label[str(int(label))]
            for label in y_test_pred[test_error_mask]
        ],
    }
)


test_confusion_pairs = (
    test_confusion_df
    .value_counts(
        [
            "actual_intent",
            "predicted_intent",
        ]
    )
    .reset_index(
        name="count",
    )
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(
        drop=True,
    )
)


print("FINAL TEST CONFUSION PAIRS")
print("=" * 70)

print(
    "Total prediction error :",
    test_error_mask.sum(),
)

print(
    "Jumlah confusion pair  :",
    len(test_confusion_pairs),
)

print()

display(
    test_confusion_pairs
)

FINAL TEST CONFUSION PAIRS
Total prediction error : 12
Jumlah confusion pair  : 6



,actual_intent,predicted_intent,count
0,return_policy,return_product_online,3
1,damaged_delivery,wrong_item,3
2,track_order,track_delivery,3
3,return_product,cancel_order,1
4,missing_item,wrong_item,1
5,request_invoice,store_location,1


In [77]:
# ============================================================
#  FINAL TEST PREDICTION ERRORS
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Cari posisi prediction yang salah
# ------------------------------------------------------------

y_test_array = np.asarray(y_test)
y_test_pred_array = np.asarray(y_test_pred)

test_error_mask = (
    y_test_array
    != y_test_pred_array
)

error_positions = np.flatnonzero(
    test_error_mask
)


# ------------------------------------------------------------
# Ambil data asli dari Test Set
# ------------------------------------------------------------

test_errors_df = (
    test_df
    .iloc[error_positions]
    .copy()
    .reset_index()
    .rename(
        columns={
            "index": "row"
        }
    )
)


# ------------------------------------------------------------
# Tambahkan actual & predicted intent
# ------------------------------------------------------------

test_errors_df["actual_intent"] = [
    id2label[int(label)]
    if int(label) in id2label
    else id2label[str(int(label))]
    for label in y_test_array[test_error_mask]
]


test_errors_df["predicted_intent"] = [
    id2label[int(label)]
    if int(label) in id2label
    else id2label[str(int(label))]
    for label in y_test_pred_array[test_error_mask]
]


# ------------------------------------------------------------
# Kolom yang ingin ditampilkan
# ------------------------------------------------------------

display_columns = [
    "row",
    "instruction",
    "actual_intent",
    "predicted_intent",
]

# Tambahkan metadata kalau tersedia
for column in [
    "category",
    "tags",
]:
    if column in test_errors_df.columns:
        display_columns.append(column)


print(
    "FINAL TEST — SELURUH PREDICTION ERROR"
)

print(
    "=" * 80
)

print(
    "Total prediction error :",
    len(test_errors_df),
)

print()


display(
    test_errors_df[
        display_columns
    ]
)

FINAL TEST — SELURUH PREDICTION ERROR
Total prediction error : 12



,row,instruction,actual_intent,predicted_intent,category,tags
0,38,is it possinle to return an item online?,return_policy,return_product_online,RETURNS,BIZ
1,171,"I'd likw to report products, will you help me?",damaged_delivery,wrong_item,DELIVERY,BCIMPZ
2,787,"I got to erturn a fucking product, I need assi...",return_product,cancel_order,RETURNS,BCWZ
3,1500,can oyu help me report products?,damaged_delivery,wrong_item,DELIVERY,BIMZ
4,2001,can i return a fucking item online,return_policy,return_product_online,RETURNS,BIQW
5,3067,wanna report a fucking miossing item i need help,missing_item,wrong_item,DELIVERY,BCQWZ
6,3166,"I got to see where ym shipment is, where can I...",track_order,track_delivery,ORDER,BCILZ
7,3512,"I want to check where my shipment is, I need a...",track_order,track_delivery,ORDER,BCL
8,3822,"I have to report products, will you help me?",damaged_delivery,wrong_item,DELIVERY,BCIMPZ
9,3967,how do I check wehere my shipment is?,track_order,track_delivery,ORDER,BILZ


In [78]:
# ============================================================
# CONFIDENCE ANALYSIS FINAL TEST ERRORS
# ============================================================

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# Helper ID -> nama intent
# ------------------------------------------------------------

def get_intent_name(class_id):
    class_id = int(class_id)

    if class_id in id2label:
        return id2label[class_id]

    if str(class_id) in id2label:
        return id2label[str(class_id)]

    return str(class_id)


# ------------------------------------------------------------
# Ambil hanya 12 teks yang salah
# ------------------------------------------------------------

error_texts = (
    test_errors_df["instruction"]
    .astype(str)
    .tolist()
)


# ------------------------------------------------------------
# Tokenisasi
# ------------------------------------------------------------

error_inputs = tokenizer(
    error_texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt",
)


device = next(
    final_model.parameters()
).device


error_inputs = {
    key: value.to(device)
    for key, value
    in error_inputs.items()
}


# ------------------------------------------------------------
# Inference 12 error
# ------------------------------------------------------------

final_model.eval()

with torch.inference_mode():

    error_logits = final_model(
        **error_inputs
    ).logits

    error_probs = torch.softmax(
        error_logits,
        dim=-1,
    )


# ------------------------------------------------------------
# Top-2 prediction
# ------------------------------------------------------------

top2_probs, top2_ids = torch.topk(
    error_probs,
    k=2,
    dim=1,
)


predicted_ids = (
    top2_ids[:, 0]
    .detach()
    .cpu()
    .numpy()
)

second_ids = (
    top2_ids[:, 1]
    .detach()
    .cpu()
    .numpy()
)


predicted_confidence = (
    top2_probs[:, 0]
    .detach()
    .cpu()
    .numpy()
)

second_best_confidence = (
    top2_probs[:, 1]
    .detach()
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Confidence terhadap label sebenarnya
# ------------------------------------------------------------

actual_ids = np.array([
    label2id[intent]
    for intent
    in test_errors_df["actual_intent"]
])


actual_ids_tensor = torch.tensor(
    actual_ids,
    dtype=torch.long,
    device=device,
)


row_indices = torch.arange(
    len(actual_ids_tensor),
    device=device,
)


actual_confidence = (
    error_probs[
        row_indices,
        actual_ids_tensor,
    ]
    .detach()
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Buat tabel confidence
# ------------------------------------------------------------

test_error_confidence_df = (
    test_errors_df[
        [
            "row",
            "instruction",
            "actual_intent",
            "predicted_intent",
        ]
    ]
    .copy()
)


test_error_confidence_df[
    "predicted_confidence"
] = predicted_confidence


test_error_confidence_df[
    "actual_confidence"
] = actual_confidence


test_error_confidence_df[
    "second_best_intent"
] = [
    get_intent_name(idx)
    for idx in second_ids
]


test_error_confidence_df[
    "second_best_confidence"
] = second_best_confidence


test_error_confidence_df[
    "confidence_margin"
] = (
    test_error_confidence_df[
        "predicted_confidence"
    ]
    -
    test_error_confidence_df[
        "second_best_confidence"
    ]
)


# ------------------------------------------------------------
# Urutkan dari error paling yakin
# ------------------------------------------------------------

test_error_confidence_df = (
    test_error_confidence_df
    .sort_values(
        "predicted_confidence",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "FINAL TEST ERROR — CONFIDENCE ANALYSIS"
)

print(
    "=" * 80
)

print(
    "Total error :",
    len(test_error_confidence_df),
)

print()


display(
    test_error_confidence_df
)

FINAL TEST ERROR — CONFIDENCE ANALYSIS
Total error : 12



,row,instruction,actual_intent,predicted_intent,predicted_confidence,actual_confidence,second_best_intent,second_best_confidence,confidence_margin
0,38,is it possinle to return an item online?,return_policy,return_product_online,0.997170,0.001377,return_policy,0.001377,0.995793
1,4321,how can I return somw products online?,return_policy,return_product_online,0.997068,0.001479,return_policy,0.001479,0.995589
2,2001,can i return a fucking item online,return_policy,return_product_online,0.995418,0.002953,return_policy,0.002953,0.992465
3,3166,"I got to see where ym shipment is, where can I...",track_order,track_delivery,0.976849,0.020755,track_order,0.020755,0.956094
4,3967,how do I check wehere my shipment is?,track_order,track_delivery,0.972298,0.025509,track_order,0.025509,0.946789
5,1500,can oyu help me report products?,damaged_delivery,wrong_item,0.957950,0.038473,damaged_delivery,0.038473,0.919477
6,171,"I'd likw to report products, will you help me?",damaged_delivery,wrong_item,0.932242,0.063440,damaged_delivery,0.063440,0.868802
7,3822,"I have to report products, will you help me?",damaged_delivery,wrong_item,0.911923,0.083181,damaged_delivery,0.083181,0.828741
8,3512,"I want to check where my shipment is, I need a...",track_order,track_delivery,0.906879,0.090145,track_order,0.090145,0.816734
9,787,"I got to erturn a fucking product, I need assi...",return_product,cancel_order,0.768137,0.092913,return_product,0.092913,0.675224


In [79]:
# ============================================================
# SUMMARY FINAL TEST ERROR CONFIDENCE
# ============================================================

total_errors = len(test_error_confidence_df)

# high-confidence error
high_conf_90 = (
    test_error_confidence_df[
        "predicted_confidence"
    ] >= 0.90
).sum()

high_conf_95 = (
    test_error_confidence_df[
        "predicted_confidence"
    ] >= 0.95
).sum()


# actual label menjadi second-best
actual_is_second = (
    test_error_confidence_df[
        "actual_intent"
    ]
    ==
    test_error_confidence_df[
        "second_best_intent"
    ]
).sum()


# low-margin / ambiguous error
low_margin_10 = (
    test_error_confidence_df[
        "confidence_margin"
    ] < 0.10
).sum()


summary_error_analysis = pd.DataFrame(
    {
        "metric": [
            "Total prediction errors",
            "Error confidence >= 90%",
            "Error confidence >= 95%",
            "Actual intent = second-best",
            "Confidence margin < 0.10",
        ],
        "count": [
            total_errors,
            high_conf_90,
            high_conf_95,
            actual_is_second,
            low_margin_10,
        ],
        "percentage": [
            100.0,
            high_conf_90 / total_errors * 100,
            high_conf_95 / total_errors * 100,
            actual_is_second / total_errors * 100,
            low_margin_10 / total_errors * 100,
        ],
    }
)


print(
    "FINAL TEST ERROR CONFIDENCE SUMMARY"
)

print("=" * 60)

display(
    summary_error_analysis
)


print()

print(
    f"High-confidence error >= 90% : "
    f"{high_conf_90}/{total_errors} "
    f"({high_conf_90 / total_errors * 100:.2f}%)"
)

print(
    f"High-confidence error >= 95% : "
    f"{high_conf_95}/{total_errors} "
    f"({high_conf_95 / total_errors * 100:.2f}%)"
)

print(
    f"Actual intent menjadi second-best : "
    f"{actual_is_second}/{total_errors} "
    f"({actual_is_second / total_errors * 100:.2f}%)"
)

print(
    f"Ambiguous error (margin < 0.10) : "
    f"{low_margin_10}/{total_errors} "
    f"({low_margin_10 / total_errors * 100:.2f}%)"
)

FINAL TEST ERROR CONFIDENCE SUMMARY


,metric,count,percentage
0,Total prediction errors,12,100.000000
1,Error confidence >= 90%,9,75.000000
2,Error confidence >= 95%,6,50.000000
3,Actual intent = second-best,11,91.666667
4,Confidence margin < 0.10,1,8.333333



High-confidence error >= 90% : 9/12 (75.00%)
High-confidence error >= 95% : 6/12 (50.00%)
Actual intent menjadi second-best : 11/12 (91.67%)
Ambiguous error (margin < 0.10) : 1/12 (8.33%)


In [80]:
# ============================================================
# PREPARE FINAL TEST REPORT DIRECTORY
# ============================================================

from pathlib import Path
import json
import pandas as pd


# cari project root secara aman
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "metrics"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "FINAL TEST REPORT DIRECTORY"
)

print("=" * 60)

print(
    "Project root :",
    PROJECT_ROOT,
)

print(
    "Report dir   :",
    REPORT_DIR,
)

print(
    "Directory exists :",
    REPORT_DIR.exists(),
)

FINAL TEST REPORT DIRECTORY
Project root : C:\Users\LENOVO\Documents\bootcamp\koding data\SupportPilot_AI_Bootcamp_KodingData
Report dir   : C:\Users\LENOVO\Documents\bootcamp\koding data\SupportPilot_AI_Bootcamp_KodingData\reports\metrics
Directory exists : True


In [81]:
# ============================================================
# FINAL DISTILBERT TEST METRICS
# ============================================================

final_test_metrics = {

    "model": "DistilBERT",

    "selection_metric": "macro_f1",

    "selection_data": "validation",

    "test_records": 4483,

    "accuracy": 0.997323,

    "macro_precision": 0.997436,

    "macro_recall": 0.997287,

    "macro_f1": 0.997325,

    "weighted_f1": 0.997326,

    "correct_predictions": 4471,

    "prediction_errors": 12,

    "inference_time_seconds": 5.4182,

    "average_inference_ms_per_sample": 1.2086,
}


final_test_metrics_df = pd.DataFrame(
    {
        "metric": final_test_metrics.keys(),
        "value": final_test_metrics.values(),
    }
)


print(
    "DISTILBERT — FINAL TEST METRICS"
)

print("=" * 60)

display(
    final_test_metrics_df
)

DISTILBERT — FINAL TEST METRICS


,metric,value
0,model,DistilBERT
1,selection_metric,macro_f1
2,selection_data,validation
3,test_records,4483
4,accuracy,0.997323
5,macro_precision,0.997436
6,macro_recall,0.997287
7,macro_f1,0.997325
8,weighted_f1,0.997326
9,correct_predictions,4471


In [82]:
# ============================================================
# SAVE FINAL TEST ARTIFACTS
# ============================================================


# ------------------------------------------------------------
# 1. Final Test metrics
# ------------------------------------------------------------

metrics_path = (
    REPORT_DIR
    / "distilbert_final_test_metrics.json"
)

with open(
    metrics_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_test_metrics,
        file,
        indent=4,
    )


# ------------------------------------------------------------
# 2. Error confidence summary
# ------------------------------------------------------------

error_summary_path = (
    REPORT_DIR
    / "distilbert_final_test_error_summary.csv"
)

summary_error_analysis.to_csv(
    error_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 3. Semua 12 prediction error
# ------------------------------------------------------------

prediction_errors_path = (
    REPORT_DIR
    / "distilbert_final_test_prediction_errors.csv"
)

test_error_confidence_df.to_csv(
    prediction_errors_path,
    index=False,
)


# ------------------------------------------------------------
# 4. Confusion pairs
# ------------------------------------------------------------

confusion_pairs_path = (
    REPORT_DIR
    / "distilbert_final_test_confusion_pairs.csv"
)

test_confusion_pairs.to_csv(
    confusion_pairs_path,
    index=False,
)


print(
    "✅ FINAL TEST ARTIFACTS BERHASIL DISIMPAN"
)

print("=" * 60)

print(
    "Metrics           :",
    metrics_path.name,
)

print(
    "Error summary     :",
    error_summary_path.name,
)

print(
    "Prediction errors :",
    prediction_errors_path.name,
)

print(
    "Confusion pairs   :",
    confusion_pairs_path.name,
)

✅ FINAL TEST ARTIFACTS BERHASIL DISIMPAN
Metrics           : distilbert_final_test_metrics.json
Error summary     : distilbert_final_test_error_summary.csv
Prediction errors : distilbert_final_test_prediction_errors.csv
Confusion pairs   : distilbert_final_test_confusion_pairs.csv


In [83]:
# ============================================================
# VERIFY SAVED ARTIFACTS
# ============================================================

artifact_paths = [
    metrics_path,
    error_summary_path,
    prediction_errors_path,
    confusion_pairs_path,
]


print(
    "FINAL ARTIFACT VERIFICATION"
)

print("=" * 60)


all_exist = True


for path in artifact_paths:

    exists = path.exists()

    size_kb = (
        path.stat().st_size / 1024
        if exists
        else 0
    )

    status = (
        "✅"
        if exists
        else "❌"
    )

    print(
        f"{status} "
        f"{path.name:<48} "
        f"{size_kb:.2f} KB"
    )

    if not exists:
        all_exist = False


print()


if all_exist:

    print(
        "✅ Semua Final Test artifact aman."
    )

else:

    print(
        "❌ Ada artifact yang belum tersimpan."
    )

FINAL ARTIFACT VERIFICATION
✅ distilbert_final_test_metrics.json               0.42 KB
✅ distilbert_final_test_error_summary.csv          0.21 KB
✅ distilbert_final_test_prediction_errors.csv      1.77 KB
✅ distilbert_final_test_confusion_pairs.csv        0.22 KB

✅ Semua Final Test artifact aman.


## Kesimpulan Final — DistilBERT

Berdasarkan seluruh proses pengembangan, evaluasi, dan analisis model, **DistilBERT** dipilih sebagai final model untuk SupportPilot AI.

### Model Selection

Pemilihan model dilakukan menggunakan **Validation Set** dengan **Macro F1** sebagai metric utama.

Hasil evaluasi pada Validation Set:

| Model | Accuracy | Macro F1 |
|---|---:|---:|
| Logistic Regression | 98.2378% | 98.2562% |
| Linear SVM | 98.7285% | 98.7429% |
| DistilBERT | **99.6431%** | **99.6491%** |

DistilBERT memberikan performa validation terbaik sehingga dipilih sebagai **final model**.

**Final Test Set tidak digunakan selama proses model selection.**

---

### Final Test Performance

Setelah final model ditentukan berdasarkan Validation Set, DistilBERT dievaluasi satu kali menggunakan **Final Test Set**.

Hasil evaluasi:

```text
Test Samples       : 4,483
Correct Prediction : 4,471
Prediction Errors  : 12

Accuracy        : 99.7323%
Macro Precision : 99.7436%
Macro Recall    : 99.7287%
Macro F1        : 99.7325%
Weighted F1     : 99.7326%
```

Model berhasil mengklasifikasikan **4,471 dari 4,483** Final Test samples dengan benar.

Hasil tersebut menunjukkan bahwa DistilBERT memiliki performa klasifikasi yang sangat tinggi dan mampu melakukan generalisasi dengan baik terhadap data yang sebelumnya tidak digunakan selama proses model development.

---

### Error Analysis

Dari **4,483 Final Test samples**, hanya terdapat **12 prediction errors**.

Beberapa confusion pair utama yang ditemukan adalah:

```text
return_policy      → return_product_online
damaged_delivery   → wrong_item
track_order        → track_delivery
```

Sebagian besar kesalahan terjadi pada intent yang memiliki kedekatan semantik atau memiliki pola kalimat yang serupa.

Contohnya, intent `track_order` dan `track_delivery` sama-sama dapat muncul pada pertanyaan customer mengenai lokasi atau status pengiriman pesanan.

---

### Confidence Analysis

Analisis confidence terhadap 12 prediction errors menghasilkan:

```text
Error confidence >= 90%     : 75.00%
Error confidence >= 95%     : 50.00%
Actual intent = second-best : 91.67%
Confidence margin < 0.10    : 8.33%
```

Sebanyak **91.67% prediction errors** memiliki actual intent sebagai kandidat dengan probabilitas tertinggi kedua.

Temuan ini menunjukkan bahwa model umumnya masih mengenali intent yang benar sebagai kandidat kuat, tetapi pada beberapa kasus terdapat batas semantik yang sangat dekat antara dua intent.

Selain itu, confidence tinggi tidak selalu menjamin bahwa prediction benar.

Karena itu, confidence score tidak digunakan sebagai satu-satunya dasar pengambilan keputusan pada production system.

---

### Confidence Policy

Untuk meningkatkan reliability pada tahap production, SupportPilot AI menggunakan **confidence-based fallback mechanism**.

Default policy:

```text
Minimum Confidence : 70%
Minimum Margin     : 10%
```

Prediction diterima apabila:

```text
confidence >= 0.70
AND
confidence_margin >= 0.10
```

Jika salah satu kondisi tidak terpenuhi, sistem mengubah final prediction menjadi:

```text
final_intent = fallback
```

Prediction tersebut kemudian dapat diarahkan untuk **human review** atau proses penanganan lainnya.

Pendekatan ini digunakan untuk mengurangi risiko penggunaan prediction ketika model tidak memiliki tingkat keyakinan yang memadai.

---

### Data Leakage Prevention

Train, Validation, dan Final Test Set dipisahkan sebelum proses model development.

Alur penggunaan dataset:

```text
Training Set
    ↓
Model Training

Validation Set
    ↓
Model Evaluation
    ↓
Model Comparison
    ↓
Final Model Selection

Final Test Set
    ↓
Final Evaluation Only
```

Model selection hanya dilakukan menggunakan **Training Set dan Validation Set**.

Final Test Set baru digunakan setelah DistilBERT ditetapkan sebagai final model.

Hasil Final Test tidak digunakan untuk:

- Hyperparameter tuning
- Retraining model
- Model selection
- Perubahan arsitektur model
- Optimasi berdasarkan Test Set

Dengan demikian, Final Test Set tetap berfungsi sebagai **unseen dataset untuk evaluasi akhir kemampuan generalisasi model**.

---

### Production Model

Final DistilBERT model disimpan pada:

```text
models/distilbert_supportpilot/best_model/
```

Model kemudian digunakan oleh production inference module:

```text
src/inference/distilbert_inference.py
```

Production inference menyediakan beberapa fungsi utama, antara lain:

- Single intent prediction
- Top-K intent prediction
- Confidence analysis
- Confidence-based fallback
- Batch prediction
- Model information

---

### Production Pipeline

Final model tidak berhenti pada tahap eksperimen di notebook, tetapi diintegrasikan ke dalam production-style application.

Arsitektur deployment:

```text
Customer Message
      ↓
Streamlit UI
      ↓
FastAPI
      ↓
DistilBERT
      ↓
Intent Prediction
      ↓
Confidence Policy
      ↓
┌───────────────┬───────────────┐
│               │               │
▼               ▼
Accepted      Fallback
Prediction    Human Review
```

Application kemudian dikemas menggunakan **Docker Compose** dengan FastAPI dan Streamlit sebagai service yang terpisah.

---

### Final Conclusion

SupportPilot AI berhasil membangun end-to-end Machine Learning system untuk **Customer Support Intent Classification** dengan **46 intent classes**.

Proses pengembangan mencakup:

1. Data Understanding
2. Data Preprocessing
3. Stratified Train / Validation / Test Split
4. Logistic Regression Baseline
5. Linear SVM
6. DistilBERT Fine-Tuning
7. Validation Model Comparison
8. Final Model Selection
9. Final Test Evaluation
10. Error Analysis
11. Confidence Analysis
12. Production Inference
13. FastAPI REST API
14. Streamlit User Interface
15. Confidence-based Fallback Mechanism
16. Automated Testing
17. Docker Deployment

DistilBERT memperoleh **Macro F1 99.6491% pada Validation Set** dan **Macro F1 99.7325% pada Final Test Set**.

Dengan demikian, tahap Machine Learning SupportPilot AI telah berhasil diselesaikan dari proses **data exploration hingga production deployment**.